## Q1. Embedding a query

In [1]:
# Sanity Check
import numpy as np
import onnxruntime
import tokenizers
from embedder import Embedder

print("onnxruntime:", onnxruntime.__version__)
print("numpy:", np.__version__)
print("Embedder imported OK")

onnxruntime: 1.27.0
numpy: 2.4.6
Embedder imported OK


In [2]:
embedder = Embedder()
v = embedder.encode("How does approximate nearest neighbor search work?")
print("vector length:", len(v))
print("v[0] =", round(float(v[0]), 4))

vector length: 384
v[0] = -0.0206


## Q2. Cosine similarity

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"loaded {len(documents)} documents") 

loaded 72 documents


In [4]:
# The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity
target = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

doc_vec = embedder.encode(target["content"])

print("query vector shape:", v.shape if hasattr(v, "shape") else len(v))
print("doc vector shape:  ", doc_vec.shape if hasattr(doc_vec, "shape") else len(doc_vec))

query vector shape: (384,)
doc vector shape:   (384,)


In [5]:
import numpy as np

q = np.asarray(v).flatten()
d = np.asarray(doc_vec).flatten()

similarity = float(np.dot(q, d))
print(f"cosine similarity: {similarity:.4f}")

cosine similarity: 0.3611


## Q3. Chunking and search by hand

In [6]:
from gitsource import chunk_documents
import numpy as np

# Chunk every page (same as homework 1)
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"{len(chunks)} chunks")

# Embed all chunks at once. encode_batch is faster than encoding one by one.
texts = [c["content"] for c in chunks]
X = embedder.encode_batch(texts)
X = np.asarray(X)
print("X shape:", X.shape)   # (n_chunks, 384)

# Score the query against every chunk via dot product.
# Vectors are normalized, so this is cosine similarity.
scores = X.dot(np.asarray(v).flatten())

# Find the winner.
best_idx = int(np.argmax(scores))
best = chunks[best_idx]

print(f"top score:  {scores[best_idx]:.4f}")
print(f"filename:   {best['filename']}")

295 chunks
X shape: (295, 384)
top score:  0.6489
filename:   02-vector-search/lessons/07-sqlitesearch-vector.md


## Q4. Vector search with minsearch

In [7]:
from minsearch import VectorSearch
import numpy as np

# Build the index. Pass the chunk vectors and the chunks as the payload.
vindex = VectorSearch()
vindex.fit(X, chunks)

# Embed the new query and search.
query = "What metric do we use to evaluate a search engine?"
q_vec = np.asarray(embedder.encode(query)).flatten()

results = vindex.search(q_vec, num_results=3)

# Print the top result's filename.
print("top result:", results[0]["filename"])

# Top 3 for sanity:
for r in results:
    print(f"  {r['filename']}")

top result: 04-evaluation/lessons/05-search-metrics.md
  04-evaluation/lessons/05-search-metrics.md
  04-evaluation/lessons/01-intro.md
  01-agentic-rag/lessons/05-search.md


## Q5. Text search vs vector search

In [8]:
from minsearch import Index
import numpy as np

# Build the keyword (text) index over the same chunks.
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

query = "How do I store vectors in PostgreSQL?"

# Vector search — embed first, then search.
q_vec = np.asarray(embedder.encode(query)).flatten()
vector_results = vindex.search(q_vec, num_results=5)

# Text search — pass the raw query.
text_results = text_index.search(query, num_results=5)

# Pull just the filenames.
vector_files = [r["filename"] for r in vector_results]
text_files   = [r["filename"] for r in text_results]

print("Vector top 5:")
for f in vector_files: print(" ", f)
print("\nText top 5:")
for f in text_files:   print(" ", f)

# Difference: in vector but not in text.
only_in_vector = set(vector_files) - set(text_files)
print("\nIn vector, not in text:", only_in_vector)

Vector top 5:
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/08-pgvector.md
  02-vector-search/lessons/08-pgvector.md

Text top 5:
  02-vector-search/lessons/02-embeddings.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md
  03-orchestration/lessons/05-rag.md
  02-vector-search/lessons/01-intro.md

In vector, not in text: {'02-vector-search/lessons/08-pgvector.md'}


## Q6. Hybrid search

In [9]:
import numpy as np

def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


query = "How do I give the model access to tools?"

# Vector search
q_vec = np.asarray(embedder.encode(query)).flatten()
vector_results = vindex.search(q_vec, num_results=5)

# Text search
text_results = text_index.search(query, num_results=5)

# Fuse
fused = rrf([vector_results, text_results])

print("Top 5 after RRF:")
for r in fused:
    print(f"  {r['filename']}  (start={r['start']})")

Top 5 after RRF:
  01-agentic-rag/lessons/13-function-calling.md  (start=4000)
  01-agentic-rag/lessons/01-intro.md  (start=2000)
  01-agentic-rag/lessons/14-agentic-loop.md  (start=0)
  04-evaluation/lessons/02-ground-truth.md  (start=1000)
  01-agentic-rag/lessons/16-other-frameworks.md  (start=0)
